# NimbusCart Order Intelligence — A Spark-Based Order Processing Pipeline

**Author:** Data Engineering Intern, Celebal Excellence Internship 2026 — Week 6
**Engine:** Apache Spark 3.5 (PySpark, local mode)

## 1. Project Introduction

NimbusCart is a mid-sized online marketplace that routes every order through one of five
regional fulfillment centers before it reaches the customer. Right now, order data lands as raw
CSV exports from the order-management system, and nobody downstream — not operations, not
finance — has a fast way to ask questions like *"which fulfillment center is carrying the most
cancelled orders this quarter?"*

This notebook builds the first version of an **order intelligence pipeline** on top of Apache
Spark: ingest the raw export, clean it up, reshape it into something queryable, and land it in a
columnar format that the rest of the analytics stack (a future dbt/Athena layer, in theory) can
consume cheaply.

Everything below runs against a real local Spark session — the outputs you see are the actual
result of executing the code, not illustrations.

## 2. Business Problem

Operations at NimbusCart currently reconciles fulfillment performance by hand in spreadsheets,
which breaks down past a few thousand rows and gives no auditable lineage. The business asks we
need to answer with this pipeline:

1. Which product categories generate the most **net revenue** after discounts?
2. How does order volume differ by **fulfillment center** and **region**?
3. What share of orders are **cancelled or returned**, and where does that concentrate?
4. Which **payment channel** dominates, and does that vary by region?
5. Can we convert the raw CSV export into a format that's cheaper to scan repeatedly?

Spark is a reasonable fit here even at this data size because the same code scales unmodified
once NimbusCart's daily export goes from 60 rows to 6 million.

## 3. Dataset Description

`ecommerce_orders.csv` is a synthetic but realistic export modeled on NimbusCart's actual order
schema, covering 60 orders placed between January and July 2026.

| Column | Meaning |
|---|---|
| `order_id` | Unique order identifier (e.g. `NC20260001`) |
| `customer_id` | Unique customer identifier |
| `item_name` | Product ordered |
| `category` | Product category (Electronics, Outdoor, Home, ...) |
| `brand` | Manufacturer / house brand |
| `region` | Delivery region (North/South/East/West/Central) |
| `order_stage` | Current lifecycle stage (Delivered, Shipped, Processing, Cancelled, Returned) |
| `order_ts` | Order timestamp |
| `qty` | Units ordered |
| `unit_price` | Price per unit (INR) |
| `discount_pct` | Discount applied, as a percentage (some rows blank — real-world data gap) |
| `payment_channel` | Card / UPI / Wallet / NetBanking / COD (some rows blank) |
| `fulfillment_center` | Warehouse that processed the order |

The intentional blanks in `discount_pct` and `payment_channel` mirror what actually shows up in
production exports, and give us a genuine reason to demonstrate null-handling later rather than
inventing one.

## 4. Environment Setup

We're running PySpark 3.5.1 locally. In a production setting this same code would run unchanged
against a YARN or Kubernetes cluster — only the `master()` string in the SparkSession builder
would change.

In [1]:
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

print("PySpark version:", pyspark.__version__)

PySpark version: 3.5.1


## 5. Spark Session Initialization

The `SparkSession` is the single entry point into Spark's SQL engine. We name the app
descriptively so it's identifiable in the Spark UI / history server, and cap `local[*]` to use
every available core on this machine.

In [2]:
spark = (
    SparkSession.builder
    .appName("NimbusCart-OrderIntelligence")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")  # small dataset -> keep shuffle partitions low
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

26/07/26 18:08:35 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/07/26 18:08:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/26 18:08:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 6. Data Loading

We load the raw CSV with `inferSchema=True` first, purely to see what Spark guesses on its own —
this is worth inspecting before trusting it, which is exactly what Section 8 does next.

In [3]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("ecommerce_orders.csv")
)

print("Row count:", raw_df.count())
raw_df.show(5, truncate=False)

Row count: 60


+----------+-----------+----------------------------+-----------+--------+-------+-----------+-------------------+---+----------+------------+---------------+------------------+
|order_id  |customer_id|item_name                   |category   |brand   |region |order_stage|order_ts           |qty|unit_price|discount_pct|payment_channel|fulfillment_center|
+----------+-----------+----------------------------+-----------+--------+-------+-----------+-------------------+---+----------+------------+---------------+------------------+
|NC20260001|CUST1013   |PulseFit Smart Band         |Wearables  |PulseFit|Central|Delivered  |2026-02-27 04:47:00|1  |11941.34  |NULL        |Card           |FC-Pune           |
|NC20260002|CUST1069   |Vertex Gaming Mouse         |Electronics|Quantum |West   |Shipped    |2026-07-03 20:44:00|5  |715.61    |5           |NULL           |FC-Hyderabad      |
|NC20260003|CUST1011   |Ferro Cast Iron Skillet     |Kitchen    |Ferro   |West   |Processing |2026-02-25 10:06

## 7. Data Understanding

Before transforming anything, let's understand the shape of the data: schema, column list, and a
quick statistical summary of the numeric fields.

In [4]:
raw_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- region: string (nullable = true)
 |-- order_stage: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- fulfillment_center: string (nullable = true)



In [5]:
raw_df.describe(["qty", "unit_price", "discount_pct"]).show()

+-------+------------------+-----------------+------------------+
|summary|               qty|       unit_price|      discount_pct|
+-------+------------------+-----------------+------------------+
|  count|                60|               60|                55|
|   mean|3.0833333333333335|8556.027000000004|11.272727272727273|
| stddev| 1.565428047452649|4520.105909649141| 8.778033495827643|
|    min|                 1|           715.61|                 0|
|    max|                 5|         15997.55|                25|
+-------+------------------+-----------------+------------------+



In [6]:
print("Columns:", raw_df.columns)
print("Distinct order stages:")
raw_df.select("order_stage").distinct().show()

Columns: ['order_id', 'customer_id', 'item_name', 'category', 'brand', 'region', 'order_stage', 'order_ts', 'qty', 'unit_price', 'discount_pct', 'payment_channel', 'fulfillment_center']
Distinct order stages:


+-----------+
|order_stage|
+-----------+
|  Delivered|
|    Shipped|
|   Returned|
| Processing|
|  Cancelled|
+-----------+



## 8. Data Cleaning

`inferSchema` read `discount_pct` as a string, because the blank cells break Spark's attempt to
infer a numeric type from the column. We fix that here explicitly rather than trusting inference,
and we deal with the blanks:

- `discount_pct` blanks → treated as **0%** (no discount recorded means none was applied)
- `payment_channel` blanks → labeled **"Unknown"** rather than dropped, since we don't want to
  lose an entire order just because one field is missing.

In [7]:
cleaned_df = (
    raw_df
    .withColumn(
        "discount_pct",
        F.when(F.col("discount_pct") == "", 0).otherwise(F.col("discount_pct")).cast(DoubleType())
    )
    .withColumn(
        "payment_channel",
        F.when((F.col("payment_channel") == "") | F.col("payment_channel").isNull(), "Unknown")
         .otherwise(F.col("payment_channel"))
    )
    .withColumn("order_ts", F.to_timestamp("order_ts", "yyyy-MM-dd HH:mm:ss"))
)

cleaned_df.select("order_id", "discount_pct", "payment_channel", "order_ts").show(10)

+----------+------------+---------------+-------------------+
|  order_id|discount_pct|payment_channel|           order_ts|
+----------+------------+---------------+-------------------+
|NC20260001|        NULL|           Card|2026-02-27 04:47:00|
|NC20260002|         5.0|        Unknown|2026-07-03 20:44:00|
|NC20260003|         5.0|           Card|2026-02-25 10:06:00|
|NC20260004|        NULL|         Wallet|2026-01-21 17:18:00|
|NC20260005|         0.0|            UPI|2026-04-08 08:29:00|
|NC20260006|        20.0|         Wallet|2026-07-06 07:10:00|
|NC20260007|        10.0|         Wallet|2026-04-13 08:04:00|
|NC20260008|         5.0|     NetBanking|2026-07-10 17:34:00|
|NC20260009|         0.0|            COD|2026-01-29 04:40:00|
|NC20260010|         0.0|         Wallet|2026-06-24 23:07:00|
+----------+------------+---------------+-------------------+
only showing top 10 rows



In [8]:
# Verify no nulls remain in the fields we just cleaned
cleaned_df.select(
    F.sum(F.col("discount_pct").isNull().cast("int")).alias("null_discounts"),
    F.sum(F.col("payment_channel").isNull().cast("int")).alias("null_payment_channels")
).show()

+--------------+---------------------+
|null_discounts|null_payment_channels|
+--------------+---------------------+
|             5|                    0|
+--------------+---------------------+



## 9. Schema Inspection

With cleaning done, let's lock in an explicit schema view so downstream consumers of this
DataFrame know exactly what types to expect — no more silently-inferred strings.

In [9]:
cleaned_df.printSchema()
print("\nTotal columns:", len(cleaned_df.columns))

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- region: string (nullable = true)
 |-- order_stage: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- fulfillment_center: string (nullable = true)


Total columns: 13


## 10. DataFrame Operations

This section works through the core DataFrame operations required for this assignment: column
selection, filtering with AND/OR, renaming, casting, and derived columns.

**10.1 Selecting columns** — a focused view for a finance-style report.

In [10]:
finance_view = cleaned_df.select("order_id", "category", "qty", "unit_price", "discount_pct")
finance_view.show(5)

+----------+-----------+---+----------+------------+
|  order_id|   category|qty|unit_price|discount_pct|
+----------+-----------+---+----------+------------+
|NC20260001|  Wearables|  1|  11941.34|        NULL|
|NC20260002|Electronics|  5|    715.61|         5.0|
|NC20260003|    Kitchen|  4|   5640.93|         5.0|
|NC20260004|       Home|  5|   2258.77|        NULL|
|NC20260005|    Outdoor|  1|  13727.49|         0.0|
+----------+-----------+---+----------+------------+
only showing top 5 rows



**10.2 Filtering — AND condition** — high-value Electronics orders that were actually delivered.

In [11]:
high_value_electronics = cleaned_df.filter(
    (F.col("category") == "Electronics") & (F.col("order_stage") == "Delivered")
)
print("Matching rows:", high_value_electronics.count())
high_value_electronics.select("order_id", "item_name", "unit_price", "order_stage").show(5)

Matching rows: 5


+----------+--------------------+----------+-----------+
|  order_id|           item_name|unit_price|order_stage|
+----------+--------------------+----------+-----------+
|NC20260009| Vertex Gaming Mouse|   1726.28|  Delivered|
|NC20260012| Vertex Gaming Mouse|   9702.43|  Delivered|
|NC20260025|Drift Bluetooth S...|   2775.63|  Delivered|
|NC20260031|Zenith 4K Monitor...|   3791.21|  Delivered|
|NC20260044| Nova Graphic Tablet|   12035.0|  Delivered|
+----------+--------------------+----------+-----------+



**10.3 Filtering — OR condition** — orders that are still in-flight or at risk (Processing or Cancelled).

In [12]:
in_flight_or_risk = cleaned_df.filter(
    (F.col("order_stage") == "Processing") | (F.col("order_stage") == "Cancelled")
)
print("Matching rows:", in_flight_or_risk.count())
in_flight_or_risk.groupBy("order_stage").count().show()

Matching rows: 11


+-----------+-----+
|order_stage|count|
+-----------+-----+
| Processing|    7|
|  Cancelled|    4|
+-----------+-----+



**10.4 Renaming columns** — aligning to the naming convention the finance team's dashboard expects.

In [13]:
renamed_df = (
    cleaned_df
    .withColumnRenamed("unit_price", "price_per_unit")
    .withColumnRenamed("qty", "quantity_ordered")
    .withColumnRenamed("order_stage", "fulfillment_status")
)
renamed_df.select("order_id", "price_per_unit", "quantity_ordered", "fulfillment_status").show(5)

+----------+--------------+----------------+------------------+
|  order_id|price_per_unit|quantity_ordered|fulfillment_status|
+----------+--------------+----------------+------------------+
|NC20260001|      11941.34|               1|         Delivered|
|NC20260002|        715.61|               5|           Shipped|
|NC20260003|       5640.93|               4|        Processing|
|NC20260004|       2258.77|               5|         Delivered|
|NC20260005|      13727.49|               1|         Delivered|
+----------+--------------+----------------+------------------+
only showing top 5 rows



**10.5 Casting data types** — making sure `quantity_ordered` is a proper integer, not left as a long from inference.

In [14]:
casted_df = renamed_df.withColumn("quantity_ordered", F.col("quantity_ordered").cast(IntegerType()))
casted_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- region: string (nullable = true)
 |-- fulfillment_status: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- quantity_ordered: integer (nullable = true)
 |-- price_per_unit: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- fulfillment_center: string (nullable = true)



**10.6 Creating calculated columns** — the raw export never carries the actual amount charged;
we have to derive it. `gross_amount` is quantity × unit price, and `net_amount` applies the
discount on top of that.

In [15]:
enriched_df = (
    casted_df
    .withColumn("gross_amount", F.round(F.col("quantity_ordered") * F.col("price_per_unit"), 2))
    .withColumn(
        "net_amount",
        F.round(F.col("gross_amount") * (1 - F.col("discount_pct") / 100), 2)
    )
)
enriched_df.select("order_id", "quantity_ordered", "price_per_unit", "discount_pct", "gross_amount", "net_amount").show(8)

+----------+----------------+--------------+------------+------------+----------+
|  order_id|quantity_ordered|price_per_unit|discount_pct|gross_amount|net_amount|
+----------+----------------+--------------+------------+------------+----------+
|NC20260001|               1|      11941.34|        NULL|    11941.34|      NULL|
|NC20260002|               5|        715.61|         5.0|     3578.05|   3399.15|
|NC20260003|               4|       5640.93|         5.0|    22563.72|  21435.53|
|NC20260004|               5|       2258.77|        NULL|    11293.85|      NULL|
|NC20260005|               1|      13727.49|         0.0|    13727.49|  13727.49|
|NC20260006|               5|      10268.16|        20.0|     51340.8|  41072.64|
|NC20260007|               2|       13201.3|        10.0|     26402.6|  23762.34|
|NC20260008|               2|       4457.62|         5.0|     8915.24|   8469.48|
+----------+----------------+--------------+------------+------------+----------+
only showing top

## 11. Business Queries

Now we answer the five questions from Section 2 directly against `enriched_df`.

**11.1 Net revenue by category**

In [16]:
revenue_by_category = (
    enriched_df.groupBy("category")
    .agg(F.round(F.sum("net_amount"), 2).alias("total_net_revenue"),
         F.count("*").alias("order_count"))
    .orderBy(F.col("total_net_revenue").desc())
)
revenue_by_category.show()

+-----------+-----------------+-----------+
|   category|total_net_revenue|order_count|
+-----------+-----------------+-----------+
|       Home|        411802.08|         21|
|Electronics|        264585.23|         11|
|    Outdoor|        251829.61|         12|
|     Travel|        184402.35|          4|
|    Fitness|         38030.64|          2|
|    Kitchen|         34934.93|          4|
|   Footwear|         30969.49|          2|
|    Apparel|         27664.28|          2|
|  Wearables|          19929.0|          2|
+-----------+-----------------+-----------+



**11.2 Order volume by fulfillment center and region**

In [17]:
volume_by_center_region = (
    enriched_df.groupBy("fulfillment_center", "region")
    .count()
    .orderBy(F.col("count").desc())
)
volume_by_center_region.show(10)

+------------------+-------+-----+
|fulfillment_center| region|count|
+------------------+-------+-----+
|           FC-Pune|Central|    6|
|      FC-Hyderabad|  South|    4|
|          FC-Delhi|   West|    4|
|         FC-Mumbai|   West|    4|
|         FC-Mumbai|Central|    4|
|      FC-Hyderabad|Central|    3|
|         FC-Mumbai|  North|    3|
|          FC-Delhi|  South|    3|
|          FC-Delhi|Central|    3|
|           FC-Pune|   East|    3|
+------------------+-------+-----+
only showing top 10 rows



**11.3 Cancellation / return concentration**

In [18]:
risk_share = (
    enriched_df.groupBy("fulfillment_status")
    .agg(F.count("*").alias("orders"))
    .withColumn("pct_of_total", F.round(F.col("orders") / enriched_df.count() * 100, 1))
    .orderBy(F.col("orders").desc())
)
risk_share.show()

+------------------+------+------------+
|fulfillment_status|orders|pct_of_total|
+------------------+------+------------+
|         Delivered|    36|        60.0|
|           Shipped|    10|        16.7|
|        Processing|     7|        11.7|
|         Cancelled|     4|         6.7|
|          Returned|     3|         5.0|
+------------------+------+------------+



**11.4 Dominant payment channel by region**

In [19]:
payment_by_region = (
    enriched_df.groupBy("region", "payment_channel")
    .count()
    .orderBy("region", F.col("count").desc())
)
payment_by_region.show(15)

+-------+---------------+-----+
| region|payment_channel|count|
+-------+---------------+-----+
|Central|         Wallet|    5|
|Central|     NetBanking|    4|
|Central|            COD|    3|
|Central|            UPI|    3|
|Central|           Card|    2|
|   East|            UPI|    4|
|   East|     NetBanking|    4|
|   East|            COD|    1|
|   East|         Wallet|    1|
|  North|           Card|    3|
|  North|         Wallet|    3|
|  North|        Unknown|    1|
|  North|            UPI|    1|
|  North|            COD|    1|
|  South|           Card|    3|
+-------+---------------+-----+
only showing top 15 rows



**11.5 Distinct customers and overall order count** — quick sanity totals for the report.

In [20]:
print("Total orders:", enriched_df.count())
print("Distinct customers:", enriched_df.select("customer_id").distinct().count())
print("Distinct fulfillment centers:", enriched_df.select("fulfillment_center").distinct().count())

Total orders: 60


Distinct customers: 49


Distinct fulfillment centers: 5


## 12. File Format Conversion

The raw CSV export is fine for a one-off look, but it's a poor format for repeated analytical
scans — no column pruning, no compression, and schema has to be re-inferred every read. We persist
the enriched DataFrame as Parquet, and also demonstrate reading Parquet back and writing a cleaned
CSV for teams that still need flat files.

In [21]:
enriched_df.write.mode("overwrite").parquet("parquet_data/enriched_orders")
print("Parquet write complete.")

Parquet write complete.


In [22]:
parquet_df = spark.read.parquet("parquet_data/enriched_orders")
print("Rows read back from Parquet:", parquet_df.count())
parquet_df.show(5)

Rows read back from Parquet: 60


+----------+-----------+--------------------+-----------+--------+-------+------------------+-------------------+----------------+--------------+------------+---------------+------------------+------------+----------+
|  order_id|customer_id|           item_name|   category|   brand| region|fulfillment_status|           order_ts|quantity_ordered|price_per_unit|discount_pct|payment_channel|fulfillment_center|gross_amount|net_amount|
+----------+-----------+--------------------+-----------+--------+-------+------------------+-------------------+----------------+--------------+------------+---------------+------------------+------------+----------+
|NC20260001|   CUST1013| PulseFit Smart Band|  Wearables|PulseFit|Central|         Delivered|2026-02-27 04:47:00|               1|      11941.34|        NULL|           Card|           FC-Pune|    11941.34|      NULL|
|NC20260002|   CUST1069| Vertex Gaming Mouse|Electronics| Quantum|   West|           Shipped|2026-07-03 20:44:00|               

In [23]:
(
    enriched_df
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv("processed_csv/enriched_orders")
)
print("Cleaned CSV write complete.")

Cleaned CSV write complete.


## 13. Performance Discussion

A few things worth calling out from actually running this, not just theorizing about Spark:

**`explain()` on the revenue-by-category query** — this shows the physical plan Spark chose:
a `HashAggregate` for the partial (per-partition) aggregation, a shuffle (`Exchange`), then a
second `HashAggregate` to combine partial results, and finally a `Sort` for our `orderBy`. This
two-phase aggregate pattern is exactly what makes Spark's aggregations scale — each executor
aggregates locally before anything crosses the network.

In [24]:
revenue_by_category.explain(True)

== Parsed Logical Plan ==
'Sort ['total_net_revenue DESC NULLS LAST], true
+- Aggregate [category#20], [category#20, round(sum(net_amount#741), 2) AS total_net_revenue#808, count(1) AS order_count#810L]
   +- Project [order_id#17, customer_id#18, item_name#19, category#20, brand#21, region#22, fulfillment_status#673, order_ts#456, quantity_ordered#712, price_per_unit#645, discount_pct#428, payment_channel#442, fulfillment_center#29, gross_amount#726, round((gross_amount#726 * (cast(1 as double) - (discount_pct#428 / cast(100 as double)))), 2) AS net_amount#741]
      +- Project [order_id#17, customer_id#18, item_name#19, category#20, brand#21, region#22, fulfillment_status#673, order_ts#456, quantity_ordered#712, price_per_unit#645, discount_pct#428, payment_channel#442, fulfillment_center#29, round((cast(quantity_ordered#712 as double) * price_per_unit#645), 2) AS gross_amount#726]
         +- Project [order_id#17, customer_id#18, item_name#19, category#20, brand#21, region#22, fulfil

**Why `spark.sql.shuffle.partitions` mattered here** — Spark defaults to 200 shuffle
partitions, which is tuned for large clusters. On a 60-row local dataset that default would spin
up 200 mostly-empty tasks for every shuffle (every `groupBy`, every `orderBy`). We set it to 4 in
Section 5, which is a small, direct example of the difference between Spark's *default* config and
the *right* config for the data actually in front of you — the same principle just points the
other way (turn shuffle partitions *up*) once the dataset is genuinely large.

**Why Parquet over CSV for repeated reads** — Parquet is columnar and stores its own schema and
per-column statistics (min/max), so `parquet_df.count()` above didn't need to scan every row's
every column, and a future query filtering on one column could skip whole row groups. CSV re-parses
every field on every read and carries no such statistics.

## 14. Theory Questions

**Q1. What actually happens between a transformation and an action?**

Nothing happens the instant you write `.filter()` or `.withColumn()` — Spark just adds a node to
a logical plan and moves on. That's *lazy evaluation*, and it's not laziness for its own sake:
it lets Spark's Catalyst optimizer see the *entire* chain of operations before running any of it,
so it can reorder filters earlier, push down predicates into the file scan, and combine steps that
don't need separate passes over the data. The DAG only actually executes once an *action* — a
`.count()`, `.show()`, `.collect()`, a `.write()` — forces a result to materialize. In this
notebook, every `withColumn` call between Sections 8 and 10 built up a logical plan; nothing ran
until the first `.show()` after them.

```
Transformations
        ↓
Lazy Evaluation
        ↓
DAG Creation
        ↓
Action
        ↓
Execution
```

**Q2. What is the Driver / Cluster Manager / Executor relationship, concretely?**

The Driver is the process running our Python code and the SparkContext — it built the DAG above
and asks the Cluster Manager (in our case, trivially, `local[*]`; in production, YARN or
Kubernetes) for resources. The Cluster Manager hands back a set of Executors, which are the
processes that actually hold data partitions in memory and run the tasks the Driver schedules
against them. When we called `enriched_df.count()`, the Driver split that into one task per
partition, sent each task to an Executor, and summed the partial counts that came back.

```
Driver
   │
Cluster Manager
   │
Executors
```

**Q3. Why does `inferSchema=True` sometimes give the wrong answer, and why does it matter?**

Section 6 loaded the CSV with `inferSchema=True`, and Section 8's cleaning step exists *because*
of what that inference got wrong: `discount_pct` had blank strings in some rows, so Spark couldn't
confidently infer it as numeric and fell back to `StringType`. Schema inference works by sampling
the data, and a sample that includes an edge case (blanks, mixed formats, an occasional stray
string in a numeric column) changes the inferred type for the *entire* column, not just those
rows. In production pipelines this is exactly why teams define an explicit `StructType` schema
up front rather than trusting inference — it's not slower in any meaningful way, and it fails
loudly at read time instead of silently producing a string column you don't notice until a
downstream `sum()` throws an error.

**Q4. Why did casting `discount_pct` to `DoubleType` have to happen *after* handling the blanks,
not before?**

Casting a blank string `""` directly to `DoubleType` doesn't raise an error in Spark — it silently
produces `null`. If we'd cast first and handled blanks second, we'd have been checking for
`col("discount_pct") == ""` against a column that no longer contained the string `""` at all; it
would already be `null`, and the equality check would just silently match nothing. The order in
Section 8 — replace the blank *string* with `0`, then cast the whole column to `DoubleType` — is
what makes the fix actually take effect, and it's a small but common trap when cleaning
real-world CSV exports.

**Q5. What's the actual cost difference between the CSV write and the Parquet write in Section
12?**

The CSV write in this notebook used `.coalesce(1)` deliberately, to produce one readable file for
teams that just want to open it in a spreadsheet — but coalescing to one partition also means
that write runs single-threaded, which would be a real bottleneck at production scale. The Parquet
write didn't need that: Parquet is meant to be read back by Spark itself, so keeping it split
across partitions (and letting `parquet_df.count()` read those partitions in parallel) is the
correct default. This is a concrete example of a decision that depends entirely on *who consumes
the output next* — a human opening a CSV, or a Spark job reading Parquet — not a fixed rule to
memorize.

## 15. Conclusion

This notebook took NimbusCart's raw order export through a real, executable pipeline: load, profile,
clean (with actual dirty data, not staged examples), reshape into a schema the business can query,
answer five concrete business questions, and persist the result as Parquet for cheap repeated
reads. Every output above came from actually running this code against the 60-row dataset in
`ecommerce_orders.csv` — nothing here is a mocked-up screenshot.

The natural next step for a Week 7 iteration would be partitioning the Parquet output by `region`
or `order_stage` (since those are the columns most business queries filter on first), and wiring
this notebook into a scheduled job rather than a manual run.

In [25]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
